# PyGeoFetch — 05: Pipelines & Scheduling

YAML data pipelines (search + download) and Python builder processing pipelines.

### What you'll learn
- Data pipeline YAML: search -> filter -> download -> export
- Processing pipeline Python builder: all 41 chainable methods
- 6 built-in processing pipeline templates
- Cron scheduling

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.core.scheduler import PipelineScheduler, Pipeline, PipelineStep
from pygeofetch.processing.pipeline import ProcessingPipeline
from pathlib import Path
import yaml

client    = PyGeoFetch(log_level="INFO")
scheduler = PipelineScheduler(engine=client)
Path("output/pipelines").mkdir(parents=True, exist_ok=True)
print("Ready")

## 1. Data Pipeline YAML

In [ ]:
# Write a YAML data pipeline
pipeline_yaml = """
name: nyc-weekly-sentinel2
schedule: "0 6 * * 1"
description: Weekly Sentinel-2 NYC

steps:
  - search:
      providers: [aws_earth, planetary_computer]
      date_range: last_7_days
      cloud_cover: 0-15
      bbox: "-74.1,40.6,-73.7,40.9"
  - filter:
      expression: "data.cloud_cover < 10"
      max_items: 5
  - download:
      parallel: 2
      output: ./output/downloads/
      bands: [B04, B08]
  - export:
      destination: ./output/cog/
"""
pipeline_path = Path("output/pipelines/nyc_weekly.yaml")
pipeline_path.write_text(pipeline_yaml)
pl = scheduler.load_pipeline(pipeline_path)
print(f"Pipeline: {pl.name!r}, Steps: {len(pl.steps)}")
for step in pl.steps:
    print(f"  [{step.step_type}]")

## 2. Processing Pipeline — Python Builder

In [ ]:
# The processing pipeline builder chains all 41 methods

result = (
    client.pipeline("ndvi-workflow")
    .atmos(method="dos1")
    .cloud_mask(method="scl")
    .clip(bbox=(-74.1, 40.6, -73.7, 40.9))
    .reproject(crs="EPSG:4326")
    .ndvi(red="B04.tif", nir="B08.tif")
    .vectorize(threshold=0.3)
    .smooth(tolerance=0.5)
    .cog(compress="deflate")
)

print(f"Pipeline: {result.name!r}  Steps: {len(result._steps)}")
for step in result._steps:
    print(f"  {step.name}")

## 3. All 41 Builder Methods

In [ ]:
# Verify all 41 builder methods compile
full = (
    client.pipeline("complete-demo")
    # Preprocessing (11)
    .atmos().cloud_mask().cloud_fill().clip().reproject().resample()
    .pansharpen().topo_correct().mosaic().composite().tile()
    # Spectral indices (17)
    .ndvi().evi().savi().ndwi().mndwi().ndbi()
    .ndsi().ndmi().nbr().dnbr().tct().pca()
    .texture().lst().albedo().band_math().stack()
    # Post-processing (9)
    .vectorize().smooth().regularize().zonal_stats()
    .buffer().centroids().add_geometry_metrics()
    .compress().cog()
    # SAR processing (4)
    .despeckle().calibrate().flood_map().coherence()
)

assert len(full._steps) == 41
print(f"All 41 builder methods verified")
print()
groups = [
    ("Preprocessing (11)", full._steps[0:11]),
    ("Spectral indices (17)", full._steps[11:28]),
    ("Post-processing (9)", full._steps[28:37]),
    ("SAR processing (4)", full._steps[37:41]),
]
for group_name, steps in groups:
    names = ", ".join(s.name for s in steps)
    print(f"  {group_name}: {names}")

## 4. Processing Pipeline YAML + from_yaml()

In [ ]:
proc_yaml = """
name: full-sentinel2-workflow
steps:
  - atmos:      {method: dos1}
  - cloud_mask: {method: scl, scl_band: SCL.tif}
  - clip:       {bbox: "-74.1,40.6,-73.7,40.9"}
  - reproject:  {crs: EPSG:4326}
  - ndvi:       {red: B04.tif, nir: B08.tif}
  - vectorize:  {threshold: 0.3}
  - cog:        {compress: deflate}
"""
proc_path = Path("output/pipelines/ndvi_proc.yaml")
proc_path.write_text(proc_yaml)
loaded = ProcessingPipeline.from_yaml(str(proc_path), engine=client)
print(f"Loaded: {loaded.name!r}, Steps: {len(loaded._steps)}")

## 5. 6 Built-in Templates

In [ ]:
templates = [
    ("ndvi",             "NDVI time series monitoring"),
    ("change_detection", "Bi-temporal change analysis"),
    ("flood_map",        "SAR-based flood mapping"),
    ("urban_mapping",    "NDBI urban/built-up analysis"),
    ("sar_analysis",     "Full SAR calibration + despeckle"),
    ("land_cover",       "Multi-index land cover"),
]

print("Built-in processing pipeline templates:")
for name, desc in templates:
    print(f"  {name:<20} {desc}")

print()
print("CLI: PyGeoFetch proc-pipeline template ndvi")
print("     PyGeoFetch proc-pipeline validate ndvi_pipeline.yaml")
print("     PyGeoFetch proc-pipeline run ndvi_pipeline.yaml --input scene.tif")

## 6. Cron Schedule Reference

In [ ]:
schedules = [
    ("0 6 * * 1",    "Every Monday 06:00 UTC"),
    ("0 6 * * *",    "Every day 06:00 UTC"),
    ("0 */6 * * *",  "Every 6 hours"),
    ("0 6 1 * *",    "First of every month"),
    ("0 6 * * 1,4",  "Monday and Thursday"),
]

print(f"{'Cron':<20} {'Meaning'}")
print("-" * 50)
for expr, meaning in schedules:
    print(f"  {expr:<18}  {meaning}")

print()
print("PyGeoFetch pipeline schedule nyc_weekly.yaml --name nyc-weekly")
print("PyGeoFetch pipeline list-scheduled")
print("PyGeoFetch pipeline history --limit 10")
print("PyGeoFetch pipeline unschedule nyc-weekly")

---
## Summary
- Data pipelines: YAML with search, filter, download, export steps
- Processing pipeline: 41 chainable builder methods verified
- Processing pipeline YAML + from_yaml() roundtrip
- 6 built-in templates
- Cron scheduling via CLI

**Next:** Notebook 06 — Real-World Workflows